In [ ]:
import sys
import h5py
import torch
H5_PATH   = # 'data_SanBernardino_new.h5'
DATA_TYPE = # 'acc' or 'disp'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

with h5py.File(H5_PATH, 'r') as f:
    names = [n.decode() for n in f['record_names'][:]]
    fs    = f.attrs['fs']
    print(f"Sampling rate: {fs} Hz")
    print(f"Total records: {len(names)}")
    layers = f[f'{DATA_TYPE}/{names[0]}/layers'][:]
    ground = f[f'{DATA_TYPE}/{names[0]}/ground'][:]
    print(f"layers shape: {layers.shape}")
    print(f"ground shape: {ground.shape}")
    print(f"\nAll records:")
    for n in names:
        print(f"  {n}")

In [ ]:
import sys
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

H5_PATH    = # 'data_SanBernardino_new.h5'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
N_FINETUNE = 5     
SEED       = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

with h5py.File(H5_PATH, 'r') as f:
    names = [n.decode() for n in f['record_names'][:]]
    fs    = float(f.attrs['fs'])

print(f"Device: {DEVICE}")
print(f"Sampling rate: {fs} Hz")
print(f"Total records: {len(names)}")

np.random.shuffle(names)
ft_names   = names[:N_FINETUNE]
test_names = names[N_FINETUNE:]
print(f"\nFine-tune ({len(ft_names)} records):")
for n in ft_names:
    print(f"  {n}")
print(f"\nTest ({len(test_names)} records):")
for n in test_names:
    print(f"  {n}")

In [ ]:
class SanBernardinoDataset(Dataset):
    def __init__(self, h5_path, record_names, data_type='acc',
                 window_size=1024, stride=512, normalize=True):
        self.h5_path     = h5_path
        self.data_type   = data_type
        self.window_size = window_size
        self.stride      = stride
        self.samples     = []
        self.scale       = 1.0

        all_layers, all_ground = [], []
        with h5py.File(h5_path, 'r') as f:
            for name in record_names:
                layers = f[f'{data_type}/{name}/layers'][:]  # [4, T]
                ground = f[f'{data_type}/{name}/ground'][:]  # [2, T]
                T = layers.shape[1]
                for start in range(0, T - window_size + 1, stride):
                    end = start + window_size
                    self.samples.append({
                        'layers': layers[:, start:end].astype(np.float32),
                        'ground': ground[:, start:end].astype(np.float32),
                    })
                all_layers.append(layers)
                all_ground.append(ground)

        if normalize:
            all_l = np.concatenate(all_layers, axis=1)
            all_g = np.concatenate(all_ground, axis=1)
            max_l = np.abs(all_l).max()
            max_g = np.abs(all_g).max()
            self.scale = float(max(max_l, max_g)) + 1e-6
            print(f"[{data_type}] scale={self.scale:.6f}  "
                  f"samples={len(self.samples)}")
            for s in self.samples:
                s['layers'] /= self.scale
                s['ground'] /= self.scale

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            'layers': torch.from_numpy(s['layers']),  # [4, 1024]
            'ground': torch.from_numpy(s['ground']),  # [2, 1024]
            'scale':  self.scale,
        }

In [ ]:
from net.Diffusion import (
    ConditionalDiffusionUNet,
    SubVPDiffusionManager,
    DDIMSampler
)

def load_pretrained_4ch(ckpt_path, device):
    model = ConditionalDiffusionUNet(
        ground_channels=2,
        condition_channels=4
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)

    model_dict    = model.state_dict()
    pretrained    = {}
    skipped_keys  = []

    for k, v in ckpt.items():
        if k not in model_dict:
            skipped_keys.append(k)
            continue
        if model_dict[k].shape == v.shape:
            pretrained[k] = v
        else:
            if model_dict[k].shape[1] < v.shape[1]:
                pretrained[k] = v[:, :model_dict[k].shape[1], :]
                print(f"[Truncated] {k}: {v.shape} → {model_dict[k].shape}")
            else:
                skipped_keys.append(k)
                print(f"[Skipped] {k}: {v.shape} vs {model_dict[k].shape}")

    model_dict.update(pretrained)
    model.load_state_dict(model_dict)

    print(f"Weights loaded, skipped {len(skipped_keys)} keys")
    return model

ACC_CKPT  = #'best_model.pth'
DISP_CKPT = #'best_model.pth'

model_acc = load_pretrained_4ch(ACC_CKPT, DEVICE)
model_disp = load_pretrained_4ch(DISP_CKPT, DEVICE)
dm      = SubVPDiffusionManager(timesteps=1000, device=DEVICE)
sampler = DDIMSampler(dm, ddim_steps=50)


In [ ]:
from net.Diffusion import freq_weighted_loss

def finetune(model, ft_dataset, device, epochs=100, lr=5e-5,
             freq_alpha=0.5, freq_fs=100.0, freq_fcut=10.0):

    TRAINABLE = ['inc', 'down1']
    for name, param in model.named_parameters():
        if any(name.startswith(k) for k in TRAINABLE):
            param.requires_grad = True
        else:
            param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")
    loader    = DataLoader(ft_dataset, batch_size=8,
                           shuffle=True, drop_last=False)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr*0.1
    )
    dm_local = SubVPDiffusionManager(timesteps=1000, device=device)

    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for batch in loader:
            ground = batch['ground'].to(device)
            layers = batch['layers'].to(device)
            t      = torch.randint(1, 1000, (ground.shape[0],), device=device)
            noisy, noise = dm_local.add_noise(ground, t)
            pred   = model(noisy, layers, t)

            loss = freq_weighted_loss(
                pred, noise,
                freq_alpha=freq_alpha,
                fs=freq_fs,
                f_cut=freq_fcut,
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            print(f"  Epoch {epoch}/{epochs}  "
                  f"loss={total_loss/len(loader):.6f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    for param in model.parameters():
        param.requires_grad = True

    return model

In [ ]:
print("Building fine-tune datasets...")
ft_ds_acc  = SanBernardinoDataset(H5_PATH, ft_names, data_type='acc')
ft_ds_disp = SanBernardinoDataset(H5_PATH, ft_names, data_type='disp')
print("\n=== Fine-tuning ACC model ===")
model_acc = finetune(
    model_acc, ft_ds_acc, DEVICE,
    epochs=100, lr=1e-4,
    freq_alpha=0.5, freq_fs=100.0, freq_fcut=10.0
)

print("\n=== Fine-tuning DISP model ===")
model_disp = finetune(
    model_disp, ft_ds_disp, DEVICE,
    epochs=100, lr=1e-4,
    freq_alpha=0.5, freq_fs=100.0, freq_fcut=10.0
)

In [ ]:
from scipy import signal
from reconstruct_full import (
    hann_overlap_add, remove_spikes, 
    compute_metrics, detect_key_segment
)

def bandpass_filter(data2d, fs=100.0, f_low=0.3, f_high=40.0, order=4):
    nyq = fs / 2.0
    b, a = signal.butter(order, [f_low/nyq, f_high/nyq], btype='band')
    out = np.zeros_like(data2d)
    for ch in range(data2d.shape[0]):
        out[ch] = signal.filtfilt(b, a, data2d[ch])
    return out


@torch.no_grad()
def evaluate_sanber(model, sampler, h5_path, test_names,
                    data_type, scale, device,
                    window_size=1024, stride=512):
    model.eval()
    all_metrics      = []
    all_metrics_seg  = []   

    with h5py.File(h5_path, 'r') as f:
        for name in tqdm(test_names, desc=f'Evaluating {data_type}'):
            layers_full = f[f'{data_type}/{name}/layers'][:]
            ground_full = f[f'{data_type}/{name}/ground'][:]
            T = layers_full.shape[1]

            pred_windows, gt_windows = [], []
            for start in range(0, T - window_size + 1, stride):
                end    = start + window_size
                layers = torch.from_numpy(
                    (layers_full[:, start:end] / scale).astype(np.float32)
                ).unsqueeze(0).to(device)
                pred = sampler.sample(
                    model, layers, device, eta=0.0, verbose=False
                )
                pred_windows.append(pred[0].cpu().numpy())
                gt_windows.append(
                    (ground_full[:, start:end] / scale).astype(np.float32)
                )

            total_len = (len(pred_windows) - 1) * stride + window_size
            pred_full = hann_overlap_add(
                pred_windows, window_size, stride, total_len
            ) * scale
            gt_full   = hann_overlap_add(
                gt_windows, window_size, stride, total_len
            ) * scale

            pred_full -= pred_full.mean(axis=1, keepdims=True)
            gt_full   -= gt_full.mean(axis=1, keepdims=True)
            pred_full  = remove_spikes(pred_full)
            pred_full = bandpass_filter(pred_full, fs=100.0,
                                        f_low=0.3, f_high=40.0)
            gt_full   = bandpass_filter(gt_full,   fs=100.0,
                                        f_low=0.3, f_high=40.0)

            m = compute_metrics(pred_full, gt_full, fs=100.0)
            m['filename'] = name
            all_metrics.append(m)
            t_start, t_end = detect_key_segment(gt_full, fs=100.0)
            i_start = int(t_start * 100)
            i_end   = int(t_end   * 100)
            pred_seg = pred_full[:, i_start:i_end]
            gt_seg   = gt_full[:,   i_start:i_end]
            m_seg = compute_metrics(pred_seg, gt_seg, fs=100.0)
            m_seg['filename'] = name
            all_metrics_seg.append(m_seg)

    def print_summary(metrics, label):
        keys = ['nrmse', 'corr_ns', 'corr_ew', 'epga_%', 'eSa_ns_%', 'eSa_ew_%']
        print(f"\n{'='*55}")
        print(f"  {data_type.upper()} {label} ({len(metrics)} records)")
        print(f"  {'Metric':<12} {'Mean':>8}  {'Std':>8}")
        print(f"  {'-'*35}")
        for key in keys:
            arr = np.array([m[key] for m in metrics])
            print(f"  {key:<12} {arr.mean():>8.4f}  {arr.std():>8.4f}")
        print(f"{'='*55}")

    print_summary(all_metrics,     'Full time history evaluation')
    print_summary(all_metrics_seg, 'Key segment evaluation')

    return all_metrics, all_metrics_seg

print("Evaluating ACC model...")
metrics_acc, metrics_acc_seg = evaluate_sanber(
    model_acc, sampler, H5_PATH, test_names,
    data_type='acc', scale=ft_ds_acc.scale, device=DEVICE
)

print("\nEvaluating DISP model...")
metrics_disp, metrics_disp_seg = evaluate_sanber(
    model_disp, sampler, H5_PATH, test_names,
    data_type='disp', scale=ft_ds_disp.scale, device=DEVICE
)

In [ ]:
with h5py.File(H5_PATH, 'r') as f:
    name = test_names[0]
    layers = f[f'acc/{name}/layers'][:]  # [4, T]
    ground = f[f'acc/{name}/ground'][:]  # [2, T]

time = np.arange(layers.shape[1]) / 100.0

fig, axs = plt.subplots(6, 1, figsize=(14, 12), sharex=True)
labels = ['CHAN004(3F-EW)', 'CHAN005(3F-NS)', 
          'CHAN007(Roof-EW)', 'CHAN008(Roof-NS)',
          'CHAN001(Ground-EW)', 'CHAN003(Ground-NS)']

for i in range(4):
    axs[i].plot(time, layers[i], lw=0.8)
    axs[i].set_ylabel(labels[i], fontsize=9)
    axs[i].grid(True, alpha=0.3)

for i in range(2):
    axs[4+i].plot(time, ground[i], color='#D95319', lw=0.8)
    axs[4+i].set_ylabel(labels[4+i], fontsize=9)
    axs[4+i].grid(True, alpha=0.3)

axs[-1].set_xlabel('Time (s)')
plt.suptitle(f'{name}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('sanber_diagnosis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Record: {name}")
print(f"layers max: {np.abs(layers).max():.4f} cm/s²")
print(f"ground max: {np.abs(ground).max():.4f} cm/s²")
print(f"layers/ground amplitude ratio: {np.abs(layers).max()/np.abs(ground).max():.3f}")

In [ ]:
from reconstruct_full import hann_overlap_add, remove_spikes, detect_key_segment

@torch.no_grad()
def visualize_prediction(model, sampler, h5_path, name, 
                         data_type, scale, device,
                         window_size=1024, stride=512):
    model.eval()
    
    with h5py.File(h5_path, 'r') as f:
        layers_full = f[f'{data_type}/{name}/layers'][:]  # [4, T]
        ground_full = f[f'{data_type}/{name}/ground'][:]  # [2, T]
    
    T = layers_full.shape[1]
    pred_windows = []
    gt_windows   = []

    for start in range(0, T - window_size + 1, stride):
        end    = start + window_size
        layers = torch.from_numpy(
            (layers_full[:, start:end] / scale).astype(np.float32)
        ).unsqueeze(0).to(device)
        pred = sampler.sample(model, layers, device, eta=0.0, verbose=False)
        pred_windows.append(pred[0].cpu().numpy())
        gt_windows.append((ground_full[:, start:end] / scale).astype(np.float32))

    total_len = (len(pred_windows) - 1) * stride + window_size
    pred_full = hann_overlap_add(pred_windows, window_size, stride, total_len) * scale
    gt_full   = hann_overlap_add(gt_windows,   window_size, stride, total_len) * scale
    pred_full -= pred_full.mean(axis=1, keepdims=True)
    gt_full   -= gt_full.mean(axis=1, keepdims=True)
    pred_full  = remove_spikes(pred_full)

    # Detect key segment
    t_start, t_end = detect_key_segment(gt_full, fs=100.0)
    i_start = int(t_start * 100)
    i_end   = int(t_end   * 100)
    time    = np.arange(total_len) / 100.0
    time_z  = time[i_start:i_end]

    C_GT   = '#1A1A2E'
    C_PRED = '#D95319'

    fig, axs = plt.subplots(2, 2, figsize=(18, 8))
    plt.rcParams.update({'font.weight':'bold', 'axes.labelsize':12,
                         'axes.titlesize':12, 'font.size':11})

    for ch, d in enumerate(['NS', 'EW']):
        # Full time history
        ax0 = axs[ch, 0]
        ax0.plot(time, gt_full[ch],   color=C_GT,   lw=1.2, label='Ground Truth')
        ax0.plot(time, pred_full[ch], color=C_PRED, lw=1.0, ls='--', label='Prediction')
        ax0.axvspan(t_start, t_end, color='#FFF3E0', alpha=0.5)
        ax0.set_title(f'{d} Full Time History')
        ax0.set_xlabel('Time (s)')
        ax0.set_ylabel('Amplitude')
        ax0.legend(fontsize=10)
        ax0.grid(True, alpha=0.3)

        # Key segment zoom
        ax1 = axs[ch, 1]
        ax1.plot(time_z, gt_full[ch][i_start:i_end],
                 color=C_GT,   lw=1.5, label='Ground Truth')
        ax1.plot(time_z, pred_full[ch][i_start:i_end],
                 color=C_PRED, lw=1.3, ls='--', label='Prediction')
        
        # Compute local PCC
        seg_gt   = gt_full[ch][i_start:i_end]
        seg_pred = pred_full[ch][i_start:i_end]
        pcc = float(np.corrcoef(seg_gt, seg_pred)[0,1]) \
              if seg_gt.std() > 1e-8 else 0.0
        corr_global = float(np.corrcoef(gt_full[ch], pred_full[ch])[0,1])
        
        ax1.text(0.02, 0.96,
                 f'Local PCC = {pcc:.3f}\nGlobal PCC = {corr_global:.3f}',
                 transform=ax1.transAxes, fontsize=10, va='top',
                 fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
        ax1.set_title(f'{d} Key Segment [{t_start:.1f}s – {t_end:.1f}s]')
        ax1.set_xlabel('Time (s)')
        ax1.set_ylabel('Amplitude')
        ax1.legend(fontsize=10)
        ax1.grid(True, alpha=0.3)

    unit = 'cm/s²' if data_type == 'acc' else 'cm'
    plt.suptitle(f'{data_type.upper()} — {name}\n(unit: {unit})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'vis_{data_type}_{name[:30]}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Image saved: vis_{data_type}_{name[:30]}.png")


with h5py.File(H5_PATH, 'r') as f:
    names = [n.decode() for n in f['record_names'][:]]

for name in test_names[:3]:
    print(f"\n=== ACC: {name} ===")
    visualize_prediction(
        model_acc, sampler, H5_PATH, name,
        data_type='acc', scale=ft_ds_acc.scale, device=DEVICE
    )
    print(f"\n=== DISP: {name} ===")
    visualize_prediction(
        model_disp, sampler, H5_PATH, name,
        data_type='disp', scale=ft_ds_disp.scale, device=DEVICE
    )

In [ ]:
acc_corr = [(m['filename'], m['corr_ns'], m['corr_ew']) 
            for m in metrics_acc_seg]
acc_sorted = sorted(acc_corr, key=lambda x: (x[1]+x[2])/2, reverse=True)

print("Top 5 ACC key-segment Corr records:")
for fname, cns, cew in acc_sorted[:5]:
    print(f"  {fname[:45]:<45} CorrNS={cns:.3f}  CorrEW={cew:.3f}")

In [ ]:
best_names = [
    'calexico_04apr2010_14607652_CE23287P',          
    'lahabra_28mar2014_15481673_ce23287p',           
    'cabazon_08may2018_38167848_ce23287p',           
    'devore_29dec2015_37507576_ce23287p',          
    'BorregoSprings_07Jul2010_CE23287P'             
]

for name in best_names:
    visualize_prediction(
        model_acc, sampler, H5_PATH, name,
        data_type='acc', scale=ft_ds_acc.scale, device=DEVICE
    )

In [ ]:
disp_corr = [(m['filename'], m['corr_ns'], m['corr_ew']) 
             for m in metrics_disp_seg]
disp_sorted = sorted(disp_corr, key=lambda x: (x[1]+x[2])/2, reverse=True)

print("Top 5 DISP key-segment Corr records:")
for fname, cns, cew in disp_sorted[:5]:
    print(f"  {fname[:45]:<45} CorrNS={cns:.3f}  CorrEW={cew:.3f}")

In [ ]:
best_disp_names = [
    'Ocotillo_14Jun2010_CE23287P',
    'BigBearLake_05Jul2014_15520985_ce23287p',
    'lahabra_28mar2014_15481673_ce23287p',
    'fontana_15jan2014_11413954_ce23287p',
    'Inglewood_17May2009_CE23287P'
]

for name in best_disp_names:
    visualize_prediction(
        model_disp, sampler, H5_PATH, name,
        data_type='disp', scale=ft_ds_disp.scale, device=DEVICE
    )